# Project Name - Apple iTunes Music Store Analysis

**Project Type = SQL Data Analysis**

**Contribution = Individual**

# Project Summary -

**The Apple iTunes Music Store Analysis project focuses on analyzing music store data using SQL queries to extract meaningful business insights. The dataset contains multiple tables such as customers, invoices, tracks, artists, albums, and genres that represent the structure of a digital music store.

In this project, SQL was used to explore customer purchasing behavior, identify top-selling artists and songs, and analyze revenue trends across different countries and cities. Various SQL techniques such as joins, aggregation functions, grouping, and subqueries were used to analyze the relationships between tables and generate insights.

The analysis helps identify the most valuable customers, the most popular music genres, and the geographic regions generating the highest revenue. These insights can help businesses improve marketing strategies, personalize music recommendations, and optimize sales performance.

This project demonstrates practical SQL skills for data analysis and provides valuable insights into customer behavior and product popularity in a digital music marketplace.**

# Problem Statement -

**The goal of this project is to analyze the Apple iTunes music store database to understand customer purchasing patterns, popular music genres, and revenue distribution across different countries and cities.

By using SQL queries, the project aims to extract meaningful insights from multiple relational tables such as customers, invoices, tracks, albums, and artists. The analysis helps identify top customers, best-selling artists, and the most popular music tracks.

These insights can help businesses make better data-driven decisions related to marketing strategies, music recommendations, and sales optimization.**

# Business Objective -

**The business objective of this project is to analyze music store data to identify customer behavior, popular artists, top-selling songs, and revenue trends across different regions. The insights generated from this analysis can help the business improve marketing strategies, increase customer engagement, and optimize music sales.**

# Lets's Begin !

**Import Libraries**

In [1]:
import pandas as pd
import sqlite3

print("Libraries imported successfully")

Libraries imported successfully


**Dataset Loading**

In [2]:
from google.colab import files
uploaded = files.upload()

Saving playlist.csv to playlist.csv
Saving playlist_track.csv to playlist_track.csv
Saving track.csv to track.csv
Saving media_type.csv to media_type.csv
Saving invoice.csv to invoice.csv
Saving invoice_line.csv to invoice_line.csv
Saving genre.csv to genre.csv
Saving employee.csv to employee.csv
Saving customer.csv to customer.csv
Saving artist.csv to artist.csv
Saving album.csv to album.csv


**Create Database**

In [3]:
conn = sqlite3.connect("itunes.db")

print("Database created successfully")

Database created successfully


**Load CSV Files into Database**

In [4]:
tables = [
    "album",
    "artist",
    "customer",
    "employee",
    "genre",
    "invoice",
    "invoice_line",
    "media_type",
    "playlist",
    "playlist_track",
    "track"
]

for table in tables:
    df = pd.read_csv(f"{table}.csv")
    df.to_sql(table, conn, if_exists="replace", index=False)

print("All tables loaded successfully")

All tables loaded successfully


**Check All Tables**

In [5]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)

,name
0,album
1,artist
2,customer
3,employee
4,genre
5,invoice
6,invoice_line
7,media_type
8,playlist
9,playlist_track


**Preview Data**

In [6]:
pd.read_sql("SELECT * FROM customer LIMIT 5;", conn)

,customer_id,first_name,last_name,company,address,city,state,country,postal_code,phone,fax,email,support_rep_id
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5
2,3,François,Tremblay,None,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,None,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,None,Ullevålsveien 14,Oslo,None,Norway,171,+47 22 44 22 22,None,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,None,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


**Customer Analytics**

Top Spending Customer-

In [7]:
query = """
SELECT c.customer_id,
       c.first_name,
       c.last_name,
       SUM(i.total) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,customer_id,first_name,last_name,total_spent
0,5,František,Wichterlová,144.54
1,6,Helena,Holý,128.70
2,46,Hugh,O'Reilly,114.84
3,58,Manoj,Pareek,111.87
4,1,Luís,Gonçalves,108.90
5,13,Fernanda,Ramos,106.92
6,34,João,Fernandes,102.96
7,42,Wyatt,Girard,99.99
8,3,François,Tremblay,99.99
9,53,Phil,Hughes,98.01


Average Customer Lifetime Value-

In [8]:
query = """
SELECT AVG(total_spent) AS avg_customer_value
FROM (
    SELECT customer_id, SUM(total) AS total_spent
    FROM invoice
    GROUP BY customer_id
);
"""
pd.read_sql(query, conn)

,avg_customer_value
0,79.820847


Repeat vs One-time customers-

In [9]:
query = """
SELECT
CASE
WHEN COUNT(invoice_id) = 1 THEN 'One-time'
ELSE 'Repeat'
END AS customer_type,
COUNT(customer_id)
FROM invoice
GROUP BY customer_id;"""
pd.read_sql(query, conn)

,customer_type,COUNT(customer_id)
0,Repeat,13
1,Repeat,11
2,Repeat,9
3,Repeat,9
4,Repeat,18
5,Repeat,12
6,Repeat,9
7,Repeat,7
8,Repeat,10
9,Repeat,12


**Sales & Revenue Analysis**

Monthly Revenue-

In [10]:
query = """
SELECT strftime('%Y-%m', invoice_date) AS month,
       SUM(total) AS revenue
FROM invoice
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, conn)

,month,revenue
0,2017-01,126.72
1,2017-02,141.57
2,2017-03,103.95
3,2017-04,142.56
4,2017-05,104.94
5,2017-06,75.24
6,2017-07,108.90
7,2017-08,88.11
8,2017-09,107.91
9,2017-10,79.20


Average Invoice Value-

In [11]:
query = """
SELECT AVG(total) AS avg_invoice_value
FROM invoice;"""
pd.read_sql(query, conn)

,avg_invoice_value
0,7.670081


**Product Analysis**

Top Revenue Track-

In [12]:
query = """
SELECT t.name,
       SUM(il.unit_price * il.quantity) AS revenue
FROM track t
JOIN invoice_line il
ON t.track_id = il.track_id
GROUP BY t.track_id
ORDER BY revenue DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,name,revenue
0,War Pigs,30.69
1,Highway Chile,13.86
2,Are You Experienced?,13.86
3,Hey Joe,12.87
4,Third Stone From The Sun,12.87
5,Put The Finger On You,12.87
6,Radio/Video,11.88
7,Love Or Confusion,11.88
8,Old School Hollywood,10.89
9,51st Anniversary,10.89


Tracks never purchased-

In [13]:
query = """
SELECT name
FROM track
WHERE track_id NOT IN (
SELECT track_id FROM invoice_line
);"""
pd.read_sql(query, conn)

,name
0,Your Time Has Come
1,Be Yourself
2,Heaven's Dead
3,Man Or Animal
4,Yesterday To Tomorrow
...,...
1692,"Concerto for Violin, Strings and Continuo in G..."
1693,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...
1694,"L'orfeo, Act 3, Sinfonia (Orchestra)"
1695,"Quintet for Horn, Violin, 2 Violas, and Cello ..."


**Artist & Genre Performance**

Top Artists-

In [14]:
query = """
SELECT ar.name,
       SUM(il.unit_price * il.quantity) AS revenue
FROM artist ar
JOIN album al ON ar.artist_id = al.artist_id
JOIN track t ON al.album_id = t.album_id
JOIN invoice_line il ON t.track_id = il.track_id
GROUP BY ar.artist_id
ORDER BY revenue DESC
LIMIT 5
"""
pd.read_sql(query, conn)

,name,revenue
0,Queen,190.08
1,Jimi Hendrix,185.13
2,Red Hot Chili Peppers,128.70
3,Nirvana,128.70
4,Pearl Jam,127.71


**Employee Performance**

In [15]:
query = """
SELECT e.first_name,
       e.last_name,
       SUM(i.total) AS revenue_generated
FROM employee e
JOIN customer c
ON e.employee_id = c.support_rep_id
JOIN invoice i
ON c.customer_id = i.customer_id
GROUP BY e.employee_id
ORDER BY revenue_generated DESC
"""
pd.read_sql(query, conn)

,first_name,last_name,revenue_generated
0,Jane,Peacock,1731.51
1,Margaret,Park,1584.00
2,Steve,Johnson,1393.92


**Geographic Trends**

In [16]:
query = """
SELECT billing_country,
       SUM(total) AS total_revenue
FROM invoice
GROUP BY billing_country
ORDER BY total_revenue DESC
"""
pd.read_sql(query, conn)

,billing_country,total_revenue
0,USA,1040.49
1,Canada,535.59
2,Brazil,427.68
3,France,389.07
4,Germany,334.62
5,Czech Republic,273.24
6,United Kingdom,245.52
7,Portugal,185.13
8,India,183.15
9,Ireland,114.84


**Q1. Who is the senior most employee based on job title?**

In [17]:
query = """
SELECT *
FROM employee
ORDER BY levels DESC
LIMIT 1;
"""
pd.read_sql(query, conn)

,employee_id,last_name,first_name,title,reports_to,levels,birthdate,hire_date,address,city,state,country,postal_code,phone,fax,email
0,9,Madan,Mohan,Senior General Manager,None,L7,26-01-1961 00:00,14-01-2016 00:00,1008 Vrinda Ave MT,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,madan.mohan@chinookcorp.com


**Q2. Which countries have the most invoices?**

In [18]:
query = """
SELECT billing_country,
       COUNT(*) AS total_invoices
FROM invoice
GROUP BY billing_country
ORDER BY total_invoices DESC;
"""
pd.read_sql(query, conn)

,billing_country,total_invoices
0,USA,131
1,Canada,76
2,Brazil,61
3,France,50
4,Germany,41
5,Czech Republic,30
6,Portugal,29
7,United Kingdom,28
8,India,21
9,Ireland,13


**Q3. What are top 3 values of total invoice?**

In [19]:
query = """
SELECT total
FROM invoice
ORDER BY total DESC
LIMIT 3;
"""
pd.read_sql(query, conn)

,total
0,23.76
1,19.80
2,19.80


**Q4. Which city has the best customers? We would like to throw a promotional Music Festival in the city we made the most money.
Write a query that returns one city that has the highest sum of invoice totals.Return both the city name & sum of all invoice totals.**

In [20]:
query = """
SELECT billing_city,
       SUM(total) AS revenue
FROM invoice
GROUP BY billing_city
ORDER BY revenue DESC
LIMIT 1;
"""
pd.read_sql(query, conn)

,billing_city,revenue
0,Prague,273.24


**Q5. Who is the best customer? The customer who has spent the most money will be declared the best customer.**

In [21]:
query = """
SELECT c.customer_id,
       c.first_name,
       c.last_name,
       SUM(i.total) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 1;
"""
pd.read_sql(query, conn)

,customer_id,first_name,last_name,total_spent
0,5,František,Wichterlová,144.54


**Q6. Write a query to return the email, first name, last name, & Genre of all Rock Music Listeners Return your list ordered alphabetically by email starting with A.**

In [22]:
query = """
SELECT DISTINCT c.email,
       c.first_name,
       c.last_name
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
JOIN invoice_line il
ON i.invoice_id = il.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN genre g
ON t.genre_id = g.genre_id
WHERE g.name = 'Rock'
ORDER BY c.email;
"""
pd.read_sql(query, conn)

,email,first_name,last_name
0,aaronmitchell@yahoo.ca,Aaron,Mitchell
1,alero@uol.com.br,Alexandre,Rocha
2,astrid.gruber@apple.at,Astrid,Gruber
3,bjorn.hansen@yahoo.no,Bjørn,Hansen
4,camille.bernard@yahoo.fr,Camille,Bernard
5,daan_peeters@apple.be,Daan,Peeters
6,diego.gutierrez@yahoo.ar,Diego,Gutiérrez
7,dmiller@comcast.com,Dan,Miller
8,dominiquelefebvre@gmail.com,Dominique,Lefebvre
9,edfrancis@yachoo.ca,Edward,Francis


**Q7. Let's invite the artists who have written the most rock music in our dataset.
Write a query to return the Artist name and total track count of the top 10 rock bands.**

In [23]:
query = """
SELECT ar.name AS artist,
       COUNT(t.track_id) AS total_tracks
FROM artist ar
JOIN album al
ON ar.artist_id = al.artist_id
JOIN track t
ON al.album_id = t.album_id
JOIN genre g
ON t.genre_id = g.genre_id
WHERE g.name = 'Rock'
GROUP BY ar.artist_id
ORDER BY total_tracks DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,artist,total_tracks
0,Led Zeppelin,114
1,U2,112
2,Deep Purple,92
3,Iron Maiden,81
4,Pearl Jam,54
5,Van Halen,52
6,Queen,45
7,The Rolling Stones,41
8,Creedence Clearwater Revival,40
9,Kiss,35


**Q8. Return all the track names that have a song length longer than the average song length.
Return the name and milliseconds for each track. Order by the song length with the longest songs listed first.**

In [24]:
query = """
SELECT name,
       milliseconds
FROM track
WHERE milliseconds >
      (SELECT AVG(milliseconds) FROM track)
ORDER BY milliseconds DESC;
"""
pd.read_sql(query, conn)

,name,milliseconds
0,Occupation / Precipice,5286953
1,Through a Looking Glass,5088838
2,"Greetings from Earth, Pt. 1",2960293
3,The Man With Nine Lives,2956998
4,"Battlestar Galactica, Pt. 2",2956081
...,...,...
489,22 Acacia Avenue,395572
490,The Unforgiven II,395520
491,The Shortest Straw,395389
492,"Concerto for Clarinet in A Major, K. 622: II. ...",394482


**Q9. Find how much amount spent by each customer on artists. Write a query to return the customer name, artist name, and total spent.**

In [25]:
query = """
SELECT c.first_name,
       c.last_name,
       ar.name AS artist,
       SUM(il.unit_price * il.quantity) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
JOIN invoice_line il
ON i.invoice_id = il.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN album al
ON t.album_id = al.album_id
JOIN artist ar
ON al.artist_id = ar.artist_id
GROUP BY c.customer_id, ar.artist_id
ORDER BY total_spent DESC;
"""
pd.read_sql(query, conn)

,first_name,last_name,artist,total_spent
0,Hugh,O'Reilly,Queen,27.72
1,Wyatt,Girard,Frank Sinatra,23.76
2,François,Tremblay,The Who,19.80
3,František,Wichterlová,Kiss,19.80
4,Helena,Holý,Red Hot Chili Peppers,19.80
...,...,...,...,...
2184,Rishabh,Mishra,Van Halen,0.99
2185,Rishabh,Mishra,Scorpions,0.99
2186,Rishabh,Mishra,Cake,0.99
2187,Rishabh,Mishra,Christopher O'Riley,0.99


**Q10. We want to find out the most popular music Genre for each country.
We determine the most popular genre as the genre with the highest amount of purchases.
Write a query that returns each country along with the top Genre. For countries where the maximum number of purchases is shared(tie) return all Genres.**

In [26]:
query = """
SELECT i.billing_country,
       g.name AS genre,
       COUNT(il.quantity) AS purchases
FROM invoice_line il
JOIN invoice i
ON il.invoice_id = i.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN genre g
ON t.genre_id = g.genre_id
GROUP BY i.billing_country, g.name
ORDER BY purchases DESC;
"""
pd.read_sql(query, conn)

,billing_country,genre,purchases
0,USA,Rock,561
1,Canada,Rock,333
2,France,Rock,211
3,Brazil,Rock,205
4,Germany,Rock,194
...,...,...,...
269,Sweden,Soundtrack,1
270,USA,TV Shows,1
271,United Kingdom,Electronica/Dance,1
272,United Kingdom,Heavy Metal,1


**Q11. Write a query that determines the customer that has spent the most on music for each country. Write a query that returns the country along with the top customer and how much they spent. For countries where the top amount spent is shared. provide all customers who spent this amount.**

In [27]:
query = """
SELECT c.country,
       c.first_name,
       c.last_name,
       SUM(i.total) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
GROUP BY c.customer_id
ORDER BY c.country, total_spent DESC;
"""
pd.read_sql(query, conn)

,country,first_name,last_name,total_spent
0,Argentina,Diego,Gutiérrez,39.60
1,Australia,Mark,Taylor,81.18
2,Austria,Astrid,Gruber,69.30
3,Belgium,Daan,Peeters,60.39
4,Brazil,Luís,Gonçalves,108.90
5,Brazil,Fernanda,Ramos,106.92
6,Brazil,Roberto,Almeida,82.17
7,Brazil,Alexandre,Rocha,69.30
8,Brazil,Eduardo,Martins,60.39
9,Canada,François,Tremblay,99.99


**Q12. Who are the most popular artists?**

In [28]:
query = """
SELECT ar.name AS artist,
       SUM(il.unit_price * il.quantity) AS revenue
FROM artist ar
JOIN album al
ON ar.artist_id = al.artist_id
JOIN track t
ON al.album_id = t.album_id
JOIN invoice_line il
ON t.track_id = il.track_id
GROUP BY ar.artist_id
ORDER BY revenue DESC
LIMIT 10;
"""
pd.read_sql(query, conn)

,artist,revenue
0,Queen,190.08
1,Jimi Hendrix,185.13
2,Red Hot Chili Peppers,128.70
3,Nirvana,128.70
4,Pearl Jam,127.71
5,Guns N' Roses,122.76
6,AC/DC,122.76
7,Foo Fighters,119.79
8,The Rolling Stones,115.83
9,Metallica,104.94


**Q13. Which is the most popular song?**

In [29]:
query = """
SELECT t.name,
       SUM(il.quantity) AS total_purchases
FROM track t
JOIN invoice_line il
ON t.track_id = il.track_id
GROUP BY t.track_id
ORDER BY total_purchases DESC
LIMIT 1;
"""
pd.read_sql(query, conn)

,name,total_purchases
0,War Pigs,31


**Q14. What are the average prices of different types of music?**

In [30]:
query = """
SELECT m.name AS media_type,
       AVG(t.unit_price) AS avg_price
FROM track t
JOIN media_type m
ON t.media_type_id = m.media_type_id
GROUP BY m.name;
"""
pd.read_sql(query, conn)

,media_type,avg_price
0,AAC audio file,0.990000
1,MPEG audio file,0.990000
2,Protected AAC audio file,0.990000
3,Protected MPEG-4 video file,1.985327
4,Purchased AAC audio file,0.990000


**Q15. What are the most popular countries for music purchases?**

In [31]:
query = """
SELECT billing_country,
       SUM(total) AS revenue
FROM invoice
GROUP BY billing_country
ORDER BY revenue DESC;
"""
pd.read_sql(query, conn)

,billing_country,revenue
0,USA,1040.49
1,Canada,535.59
2,Brazil,427.68
3,France,389.07
4,Germany,334.62
5,Czech Republic,273.24
6,United Kingdom,245.52
7,Portugal,185.13
8,India,183.15
9,Ireland,114.84


# Key Insights

**

1. The city generating the highest revenue indicates a strong customer base and could be a potential location for marketing events.

2. A small group of customers contributes a large portion of the revenue.

3. Rock music appears to be one of the most popular genres based on customer purchases.

4. Certain artists generate significantly higher revenue compared to others.

5. Some countries contribute more revenue per customer than others.**

# Conclusion

##

In this project, we analyzed the Apple iTunes music store dataset using SQL queries. The analysis helped identify top customers, popular music genres, best-selling artists, and revenue distribution across different countries and cities.

These insights can help businesses improve marketing strategies, optimize product offerings, and enhance customer engagement.

Overall, this project demonstrates how SQL can be used to analyze relational datasets and generate meaningful business insights.

# Create SQL File

In [32]:
sql_queries = """
-- Q1 Senior most employee
SELECT *
FROM employee
ORDER BY levels DESC
LIMIT 1;

-- Q2 Countries with most invoices
SELECT billing_country, COUNT(*) AS total_invoices
FROM invoice
GROUP BY billing_country
ORDER BY total_invoices DESC;

-- Q3 Top 3 invoice totals
SELECT total
FROM invoice
ORDER BY total DESC
LIMIT 3;

-- Q4 City with highest revenue
SELECT billing_city, SUM(total) AS total_revenue
FROM invoice
GROUP BY billing_city
ORDER BY total_revenue DESC
LIMIT 1;

-- Q5 Best customer
SELECT c.customer_id, c.first_name, c.last_name,
SUM(i.total) AS total_spent
FROM customer c
JOIN invoice i ON c.customer_id = i.customer_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 1;

-- Q6 Rock music listeners
SELECT DISTINCT c.email,
       c.first_name,
       c.last_name
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
JOIN invoice_line il
ON i.invoice_id = il.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN genre g
ON t.genre_id = g.genre_id
WHERE g.name = 'Rock'
ORDER BY c.email;

-- Q7 Top 10 Rock artists
SELECT ar.name AS artist,
       COUNT(t.track_id) AS total_tracks
FROM artist ar
JOIN album al
ON ar.artist_id = al.artist_id
JOIN track t
ON al.album_id = t.album_id
JOIN genre g
ON t.genre_id = g.genre_id
WHERE g.name = 'Rock'
GROUP BY ar.artist_id
ORDER BY total_tracks DESC
LIMIT 10;

-- Q8 Tracks longer than average
SELECT name,
       milliseconds
FROM track
WHERE milliseconds >
      (SELECT AVG(milliseconds) FROM track)
ORDER BY milliseconds DESC;

-- Q9 Amount spent by each customer on artists
SELECT c.first_name,
       c.last_name,
       ar.name AS artist,
       SUM(il.unit_price * il.quantity) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
JOIN invoice_line il
ON i.invoice_id = il.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN album al
ON t.album_id = al.album_id
JOIN artist ar
ON al.artist_id = ar.artist_id
GROUP BY c.customer_id, ar.artist_id
ORDER BY total_spent DESC;

-- Q10 Most popular genre for each country
SELECT i.billing_country,
       g.name AS genre,
       COUNT(il.quantity) AS purchases
FROM invoice_line il
JOIN invoice i
ON il.invoice_id = i.invoice_id
JOIN track t
ON il.track_id = t.track_id
JOIN genre g
ON t.genre_id = g.genre_id
GROUP BY i.billing_country, g.name
ORDER BY purchases DESC;

-- Q11 Top customer by country
SELECT c.country,
       c.first_name,
       c.last_name,
       SUM(i.total) AS total_spent
FROM customer c
JOIN invoice i
ON c.customer_id = i.customer_id
GROUP BY c.customer_id
ORDER BY c.country, total_spent DESC;

-- Q12 Most popular artists by revenue
SELECT ar.name AS artist,
       SUM(il.unit_price * il.quantity) AS revenue
FROM artist ar
JOIN album al
ON ar.artist_id = al.artist_id
JOIN track t
ON al.album_id = t.album_id
JOIN invoice_line il
ON t.track_id = il.track_id
GROUP BY ar.artist_id
ORDER BY revenue DESC
LIMIT 10;

-- Q13 Most popular song
SELECT t.name,
       SUM(il.quantity) AS total_purchases
FROM track t
JOIN invoice_line il
ON t.track_id = il.track_id
GROUP BY t.track_id
ORDER BY total_purchases DESC
LIMIT 1;

-- Q14 Average price by media type
SELECT m.name AS media_type,
       AVG(t.unit_price) AS avg_price
FROM track t
JOIN media_type m
ON t.media_type_id = m.media_type_id
GROUP BY m.name;

-- Q15 Revenue by country
SELECT billing_country,
       SUM(total) AS revenue
FROM invoice
GROUP BY billing_country
ORDER BY revenue DESC;

"""

with open("itunes_music_analysis.sql", "w") as f:
    f.write(sql_queries)

print("SQL file created successfully")

SQL file created successfully


In [34]:
from google.colab import files
files.download("itunes_music_analysis.sql")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>